# 获取聚宽因子库所有因子

In [5]:
from jqfactor import get_all_factors 

all_factors = get_all_factors()
all_factors.head(10)

,factor,factor_intro,category,category_intro
0,administration_expense_ttm,管理费用TTM,basics,基础科目及衍生类因子
1,asset_impairment_loss_ttm,资产减值损失TTM,basics,基础科目及衍生类因子
2,cash_flow_to_price_ratio,现金流市值比,basics,基础科目及衍生类因子
3,circulating_market_cap,流通市值,basics,基础科目及衍生类因子
4,EBIT,息税前利润,basics,基础科目及衍生类因子
5,EBITDA,息税折旧摊销前利润,basics,基础科目及衍生类因子
6,financial_assets,金融资产,basics,基础科目及衍生类因子
7,financial_expense_ttm,财务费用TTM,basics,基础科目及衍生类因子
8,financial_liability,金融负债,basics,基础科目及衍生类因子
9,goods_sale_and_service_render_cash_ttm,销售商品提供劳务收到的现金,basics,基础科目及衍生类因子


In [7]:
all_factors['category'].value_counts()

quality      71
basics       37
emotion      36
momentum     34
style        30
technical    16
pershare     15
risk         12
growth        9
Name: category, dtype: int64

In [8]:
all_factors[all_factors['category']=='style']

,factor,factor_intro,category,category_intro
214,average_share_turnover_annual,年度平均月换手率,style,风险因子 - 风格因子
215,average_share_turnover_quarterly,季度平均平均月换手率,style,风险因子 - 风格因子
216,beta,BETA,style,风险因子 - 风格因子
217,book_leverage,账面杠杆,style,风险因子 - 风格因子
218,book_to_price_ratio,市净率因子,style,风险因子 - 风格因子
219,cash_earnings_to_price_ratio,现金流量市值比,style,风险因子 - 风格因子
220,cube_of_size,市值立方因子,style,风险因子 - 风格因子
221,cumulative_range,收益离差,style,风险因子 - 风格因子
222,daily_standard_deviation,日收益率标准差,style,风险因子 - 风格因子
223,debt_to_assets,资产负债率,style,风险因子 - 风格因子


#  行业

In [ ]:
# 获取行业板块成分股
get_industry_stocks(industry_code, date=None)


In [50]:
from jqdata import *
get_industries('zjw', date=None).to_csv('zjw_index.csv')

# 国盛证券 因子

In [1]:
import datetime
import pandas as pd
import numpy as np 
#import json

In [4]:
# class G(object):
#     def __init__(self):
#         pass
# g = G()
# # g.benchmark = '000001.XSHG' # 000300.XSHG (HS300) 000905.XSHG (ZZ500)
# # g.sample_windows = 252 # 样本窗口长度
# g.buy_list = []
# g.stock_num = 5   # 持仓最大股票数

trade_days = [d.date() for d in get_price('000300.XSHG', 
                                          start_date='2015-01-01', 
                                          end_date='2023-07-10', 
                                          frequency='daily', 
                                          fields=None, 
                                          skip_paused=True, 
                                          fq='pre').index]
trade_days

[datetime.date(2015, 1, 5),
 datetime.date(2015, 1, 6),
 datetime.date(2015, 1, 7),
 datetime.date(2015, 1, 8),
 datetime.date(2015, 1, 9),
 datetime.date(2015, 1, 12),
 datetime.date(2015, 1, 13),
 datetime.date(2015, 1, 14),
 datetime.date(2015, 1, 15),
 datetime.date(2015, 1, 16),
 datetime.date(2015, 1, 19),
 datetime.date(2015, 1, 20),
 datetime.date(2015, 1, 21),
 datetime.date(2015, 1, 22),
 datetime.date(2015, 1, 23),
 datetime.date(2015, 1, 26),
 datetime.date(2015, 1, 27),
 datetime.date(2015, 1, 28),
 datetime.date(2015, 1, 29),
 datetime.date(2015, 1, 30),
 datetime.date(2015, 2, 2),
 datetime.date(2015, 2, 3),
 datetime.date(2015, 2, 4),
 datetime.date(2015, 2, 5),
 datetime.date(2015, 2, 6),
 datetime.date(2015, 2, 9),
 datetime.date(2015, 2, 10),
 datetime.date(2015, 2, 11),
 datetime.date(2015, 2, 12),
 datetime.date(2015, 2, 13),
 datetime.date(2015, 2, 16),
 datetime.date(2015, 2, 17),
 datetime.date(2015, 2, 25),
 datetime.date(2015, 2, 26),
 datetime.date(2015, 2, 2

In [5]:
# 过滤【次新 + 科创北交 + ST退市 + 停牌】
def filter_stocks(all_security_df, yesterday, current_dt):
    all_security_df['index'] = all_security_df.index

    # 过滤次新股
    filter_fn = lambda t: (yesterday - t) >= datetime.timedelta(days=375)
    all_security_df = all_security_df[all_security_df['start_date'].apply(filter_fn)]

    # 过滤科创北交
    filter_fn = lambda t: t[0] != '4' and t[0] != '8' and t[:2] != '68'
    all_security_df = all_security_df[all_security_df['index'].apply(filter_fn)]

    # 过滤ST及其他具有退市标签的股票
    st_df = get_extras('is_st', all_security_df.index, start_date=current_dt, end_date=current_dt, df=True).T
    st_list = st_df[st_df==True].dropna().index
    filter_fn = lambda t: ("ST" in t or "*" in t or "退" in t)
    all_security_df = all_security_df[(~all_security_df['index'].isin(st_list)) & (~all_security_df['display_name'].apply(filter_fn))]
    
    # 过滤停牌
    susp_df = get_price(all_security_df.index.tolist(), \
                        end_date=current_dt, count=1, frequency='daily', \
                        fields='paused', panel=False)
    unsusp_stocks = susp_df[susp_df['paused'] < 1]["code"].tolist() # 得到当日未停牌股票代码list:
    days=7 # 过滤出前7天没有停牌过的Stocks
    feasible_stocks=[s for s in unsusp_stocks 
                     if sum(attribute_history(s, days, unit='1d',fields=('paused'),skip_paused=False))[0]==0]
    return feasible_stocks


## 1_波动性因子

### ivol_60
- 1  ivol          波动性 特质波动率（Fama French 回归的残差波动率）
### ivr_60
- 2  ivr           波动性 特异度（1-Fama French 回归的拟合优度）
### ret_std_20
- 3  return_std_1m 波动性 一个月收益率波动率（单位成交量下价格的波动）

In [4]:
import statsmodels.api as sm
from statsmodels import regression

def get_fama_french_reg_res(all_security, yesterday, current_dt, benchmark, samp_window):
    q = query(
        valuation.code, 
        valuation.market_cap, # 市值
        (balance.total_owner_equities/valuation.market_cap/100000000.0).label("BTM"), # BP
        ).filter(valuation.code.in_(all_security))
    df = get_fundamentals(q, yesterday)
    
    pool_size = len(all_security)
    valid_size = pool_size // 3
    # 升序排序
    B = df.sort_values("market_cap")["code"][:valid_size]
    S = df.sort_values("market_cap")["code"][-valid_size:]
    L = df.sort_values('BTM')['code'][:valid_size]
    H = df.sort_values('BTM')['code'][-valid_size:]
    
    prices_df = get_price(all_security, end_date=yesterday, fields="close", count=samp_window, 
                          frequency="daily", panel=False)    
    close_df = pd.pivot_table(prices_df, index='time', columns='code', values='close')
    r_df = np.diff(np.log(close_df), axis=0) + 0 * close_df[1:] # 计算对数收益率
    SMB = (sum(r_df[B].T) - sum(r_df[S].T)) / len(S)
    HML = (sum(r_df[H].T) - sum(r_df[L].T)) / len(L)

    rf = get_bond_average_rate(end_date=yesterday, count=samp_window)
    dp = get_price(benchmark, end_date=yesterday, fields="close", count=samp_window, 
                   frequency='daily', panel=False)
    RM_rf = (np.diff(np.log(dp), axis=0) + 0*dp[1:])["close"] - rf
    
    X = pd.DataFrame({
            "RM_rf": RM_rf,
            "SMB":   SMB,
            "HML":   HML
        })
    X = sm.add_constant(array(X))
    
    errors = []
    ivols = []
    rsquares = []
    for t_stock in all_security:
        Y = r_df[t_stock]-rf
        if len(Y)>1:
            results = regression.linear_model.OLS(Y, X).fit()
            errors.append(results.params[0])  # 残差
            ivol = (Y - X[:,1]*results.params[1] - X[:,2]*results.params[2] - X[:,3]*results.params[3]).std()
            ivols.append(ivol) # 残差波动性
            rsquares.append(results.rsquared) # RSquared            
        else:
            errors.append(float('nan'))   # 残差
            ivols.append(float('nan'))    # 残差波动性
            rsquares.append(float('nan')) # RSquared
    
    score_df = pd.DataFrame()
    score_df['code']=all_security
    score_df['error'] = errors
    score_df['ivol'] = ivols
    score_df['Rsquare'] = rsquares
    score_df['ivr'] = 1-score_df['Rsquare']
    score_df = score_df.set_index('code')
    return score_df

# 计算bound_code的日均利率  "511010.XSHG"是国债ETF
def get_bond_average_rate(bond_code="511010.XSHG", end_date=None, count=252):
    end_date = end_date if end_date else datetime.datetime.now()
    df = get_price(bond_code, end_date=end_date, count=count, frequency='daily', fields=['close'])
    return (np.diff(np.log(df), axis=0) + 0*df[1:])["close"]


In [5]:
for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1 and str(current_dt)>'2022-08-22': 
        print('Yesterday={}, Today={}'.format(yesterday, current_dt))
        
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)               
        
        # Calculate Indicators
        score252_df = get_fama_french_reg_res(feasible_security, yesterday, current_dt, 
                                           benchmark='000001.XSHG', samp_window=252)
        score20_df = get_fama_french_reg_res(feasible_security, yesterday, current_dt, 
                                           benchmark='000001.XSHG', samp_window=20)
        
        prices_df = get_price(all_security, end_date=yesterday, fields="close", count=21, frequency="daily", panel=False)    
        close_df = pd.pivot_table(prices_df, index='time', columns='code', values='close')
        r_df = np.diff(np.log(close_df), axis=0) + 0 * close_df[1:] # 计算对数收益率
        
        # Save File
        res_df = pd.DataFrame()
        res_df['ivr_1y'] = score252_df['ivr']
        res_df['ivol_1y'] = score252_df['ivol']
        res_df['ivr_1m'] = score20_df['ivr']
        res_df['ivol_1m'] = score20_df['ivol']
        res_df['ret_std_1m'] = r_df[-20:].std()
        res_df.to_csv('data/1_Volability/{}.csv'.format(str(current_dt)))
        

Yesterday=2022-08-26, Today=2022-08-29
Yesterday=2022-09-02, Today=2022-09-05
Yesterday=2022-09-09, Today=2022-09-13
Yesterday=2022-09-16, Today=2022-09-19
Yesterday=2022-09-23, Today=2022-09-26
Yesterday=2022-09-30, Today=2022-10-10
Yesterday=2022-10-14, Today=2022-10-17
Yesterday=2022-10-21, Today=2022-10-24
Yesterday=2022-10-28, Today=2022-10-31
Yesterday=2022-11-04, Today=2022-11-07
Yesterday=2022-11-11, Today=2022-11-14
Yesterday=2022-11-18, Today=2022-11-21
Yesterday=2022-11-25, Today=2022-11-28
Yesterday=2022-12-02, Today=2022-12-05
Yesterday=2022-12-09, Today=2022-12-12
Yesterday=2022-12-16, Today=2022-12-19
Yesterday=2022-12-23, Today=2022-12-26
Yesterday=2022-12-30, Today=2023-01-03
Yesterday=2023-01-06, Today=2023-01-09
Yesterday=2023-01-13, Today=2023-01-16
Yesterday=2023-01-20, Today=2023-01-30
Yesterday=2023-02-03, Today=2023-02-06
Yesterday=2023-02-10, Today=2023-02-13
Yesterday=2023-02-17, Today=2023-02-20
Yesterday=2023-02-24, Today=2023-02-27
Yesterday=2023-03-03, Tod

## 2_成长

### yoy_bps 成长 (当期 BPS-去年同期 BPS)/去年同期 BPS 绝对值,其中 BPS 表示每股帐面价值
### yoy_orps 成长 每股营业收入同比增长率
### yoy_eps 成长 每股收益同比增速
### yoy_tot_equity 成长 (当期股东权益-去年同期股东权益)/去年同期股东权益绝对值
### yoy_total_assets 成长 (当期总资产-去年同期总资产)/去年同期总资产绝对值
### yoy_orps_q 成长 单季度每股营业收入同比增长率
### yoy_eps_q 成长 单季度每股收益同比增速
### yoy_ocfps 成长 每股经营现金流同比增速
### yoy_ocfps_q 成长 单季度每股经营现金流同比增速
### yoy_np 成长 净利润同比增速
### yoy_np_q 成长 单季度净利润同比增速
### yoy_or 成长 营业收入同比增速
### yoy_or_q 成长 单季度营业收入同比增速
### yoy_op 成长 经营利润同比增速
### yoy_op_q 成长 单季度经营利润同比增速
### yoy_ocf 成长 经营现金流同比增速
### yoy_ocf_q 成长 单季度经营现金流同比增速
### yoy_roe 成长 (当期 ROE-去年同期 ROE)/去年同期 ROE 绝对值

In [4]:
from jqdata import get_history_fundamentals
from jqdata import get_valuation

In [5]:
for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1:
        last_year_yesterday = '{}{}'.format(int(str(yesterday)[:4])-1, str(yesterday)[4:])
        print('==== Yesterday={}, Today={} ====='.format(yesterday, current_dt))
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)
        
    # Calculate Indicators
        fields = [
            balance.total_assets,        # 总资产
            balance.total_liability,     # 总负债
            balance.total_owner_equities,# 总所有者权益

            income.total_operating_revenue,# 营业总收入(元) 具体核算范围和方法参见上市公司定期报告
            income.operating_revenue,      # 营业收入(元)   具体核算范围和方法参见上市公司定期报告

            income.total_operating_cost,# 营业总成本(元) 营业总成本=主营业务成本+其他业务成本+利息支出+手续费及佣金支出+退保金+赔付支出净额+提取保险合同准备金净额+保单红利支出+分保费用+营业税金及附加+销售费用+管理费用+财务费用+资产减值损失+其他
            income.operating_cost,      # 营业成本(元)   营业成本，也称运营成本。是指企业所销售商品或者提供劳务的成本。营业成本应当与所销售商品或者所提供劳务而取得的收入进行配比。
            income.operating_profit,    # 营业利润(元)   营业利润是企业最基本经营活动的成果，也是企业一定时期获得利润中最主要、最稳定的来源。2006年财政部颁布的新企业会计准则-30号财务报表列报中已对营业利润进行了调整，将投资收益调入营业利润，同时取消了主营业务利润和其他业务利润的提法，补贴收入被并入营业外收入，营业利润减营业外收支调整即得到利润总额。

            income.total_profit,        # 利润总额(元)   利润总额指企业在生产经营过程中各种收入扣除各种耗费后的盈余，反映企业在报告期内实现的盈亏总额。
            income.income_tax_expense,  # 所得税费用(元) 所得税费用是指企业经营利润应交纳的所得税。“所得税费用”，核算企业负担的所得税，是损益类科目；这一般不等于当期应交所得税，因为可能存在“暂时性差异”。如果只有永久性差异，则等于当期应交所得税。
            income.net_profit,          # 净利润                

            income.interest_income,     # 利息收入(元) 利息收入是指纳税人购买各种债券等有价证券的利息，外单位欠款付给的利息以及其他利息收入。包括：购买各种债券等有价证券的利息，如购买国库券，重点企业建设债券、国家保值公债以及政府部门和企业发放的各类有价证券；企业各项存款所取得的利息外单位欠本企业款而取得的利息；其他利息收入等。
            income.interest_expense,    # 利息支出(元) 利息支出是指临时借款的利息支出。在以收付实现制作为记帐基础的前提条件下，所谓支出应以实际支付为标准，即资金流出，标志着现金、银行存款的减少。就利息支出而言、给个人帐户计息，其资金并没有流出，现金、银行存款并没有减少，因此，给个人计息不应作为利息支出列支。

            indicator.eps,              # 每股收益EPS(元)	每股收益(摊薄)＝净利润/期末股本；分子从单季利润表取值，分母取季度末报告期股本值；净利润指归属于母公司股东的净利润(元)
            indicator.roe,              # 净资产收益率ROE(%) => 归属于母公司股东的净利润*2/（期初归属于母公司股东的净资产+期末归属于母公司股东的净资产）
        ]

        # ================ YOY of Q indis ================
        q_df = get_history_fundamentals(feasible_security, fields, watch_date=yesterday, count=5, 
                                        interval='1q', stat_by_year=False)
        q_df['interest_income'].fillna(0, inplace=True)
        q_df['interest_expense'].fillna(0, inplace=True)
        q_df['orps'] = q_df['operating_revenue']/q_df['total_owner_equities']
        q_df['ocf'] = q_df['operating_profit']+q_df['interest_expense']
        q_df['ocfps'] = q_df['ocf']/q_df['total_owner_equities']
        
        min_q_dates = q_df.groupby('code')['statDate'].min()
        pre_q_indi_df = pd.merge(q_df, min_q_dates.reset_index(), how='right', on=['code','statDate']).set_index('code')
        max_q_dates = q_df.groupby('code')['statDate'].max()
        curr_q_indi_df = pd.merge(q_df, max_q_dates.reset_index(), how='right', on=['code','statDate']).set_index('code')
        print('curr_q_indi_df:{}, pre_q_indi_df:{}'.format(curr_q_indi_df.shape, pre_q_indi_df.shape))
        
        # 7个 Q指标 yoy
        yoy_orps_q_df = curr_q_indi_df['orps']/pre_q_indi_df['orps'] - 1    # yoy_orps_q  成长 单季度每股营业收入同比增长率
        yoy_eps_q_df = curr_q_indi_df['eps']/pre_q_indi_df['eps'] - 1       # yoy_eps_q   成长 单季度每股收益同比增速
        yoy_ocfps_q_df = curr_q_indi_df['ocfps']/pre_q_indi_df['ocfps'] - 1 # yoy_ocfps_q 成长 单季度每股经营现金流同比增速

        yoy_np_q_df = curr_q_indi_df['net_profit']/pre_q_indi_df['net_profit'] - 1               # yoy_np_q  成长 单季度净利润同比增速
        yoy_or_q_df = curr_q_indi_df['operating_revenue']/pre_q_indi_df['operating_revenue'] - 1 # yoy_or_q  成长 单季度营业收入同比增速
        yoy_op_q_df = curr_q_indi_df['operating_profit']/pre_q_indi_df['operating_profit'] - 1   # yoy_op_q  成长 单季度经营利润同比增速
        yoy_ocf_q_df = curr_q_indi_df['ocf']/pre_q_indi_df['ocf'] - 1                            # yoy_ocf_q 成长 单季度经营现金流同比增速

        # ================ YOY of Y indis ================
        y_df = get_history_fundamentals(feasible_security, fields, watch_date=yesterday, count=2, 
                                        interval='1y', stat_by_year=True)
        y_df['interest_income'].fillna(0, inplace=True)
        y_df['interest_expense'].fillna(0, inplace=True)        
        y_df['orps'] = y_df['operating_revenue']/y_df['total_owner_equities']
        y_df['ocf'] = y_df['operating_profit']+y_df['interest_expense']
        y_df['ocfps'] = y_df['ocf']/y_df['total_owner_equities']

        min_y_dates = y_df.groupby('code')['statDate'].min()
        pre_y_indi_df = pd.merge(y_df, min_y_dates.reset_index(), how='right', on=['code','statDate'])
        max_y_dates = y_df.groupby('code')['statDate'].max()
        curr_y_indi_df = pd.merge(y_df, max_y_dates.reset_index(), how='right', on=['code','statDate'])
        
        pre_v_df = get_valuation(feasible_security, end_date=last_year_yesterday, count=1, fields='circulating_market_cap')
        curr_v_df = get_valuation(feasible_security, end_date=yesterday,        count=1, fields='circulating_market_cap')

        pre_y_indi_df = pd.merge(pre_y_indi_df, pre_v_df[['code','circulating_market_cap']], on='code', how='left').set_index('code')
        pre_y_indi_df['bps'] = pre_y_indi_df['total_owner_equities']/pre_y_indi_df['circulating_market_cap']
        curr_y_indi_df = pd.merge(curr_y_indi_df, curr_v_df[['code','circulating_market_cap']], on='code', how='left').set_index('code')
        curr_y_indi_df['bps'] = curr_y_indi_df['total_owner_equities']/curr_y_indi_df['circulating_market_cap']        
        print('curr_y_indi_df:{}, pre_y_indi_df:{}'.format(curr_y_indi_df.shape, pre_y_indi_df.shape))
        
        # Calc Indicators
        yoy_tot_equity_df = (curr_y_indi_df['total_owner_equities']-pre_y_indi_df['total_owner_equities'])/abs(pre_y_indi_df['total_owner_equities']) - 1 # yoy_tot_equity    成长 股东权益同比增速  (当期股东权益-去年同期股东权益)/去年同期股东权益绝对值
        yoy_total_assets_df = (curr_y_indi_df['total_assets']-pre_y_indi_df['total_assets'])/abs(pre_y_indi_df['total_assets']) - 1 # yoy_total_assets  成长 总资产同比增速    (当期总资产-去年同期总资产)/去年同期总资产绝对值
        yoy_bps_df = (curr_y_indi_df['bps']-pre_y_indi_df['bps'])/abs(pre_y_indi_df['bps']) - 1 # yoy_bps           成长 每股帐面价值同比增长率 (当期 BPS-去年同期 BPS)/去年同期 BPS 绝对值,其中 BPS 表示每股帐面价值
        yoy_orps_df = curr_y_indi_df['orps']/pre_y_indi_df['orps'] - 1    # yoy_orps_q   成长 单季度每股营业收入同比增长率
        yoy_eps_df = curr_y_indi_df['eps']/pre_y_indi_df['eps'] - 1       # yoy_eps_q    成长 单季度每股收益同比增速
        yoy_ocfps_df = curr_y_indi_df['ocfps']/pre_y_indi_df['ocfps'] - 1 # yoy_ocfps_q  成长 单季度每股经营现金流同比增速
        yoy_np_df = curr_y_indi_df['net_profit']/pre_y_indi_df['net_profit'] -1 # yoy_np 成长 净利润同比增速
        yoy_or_df = curr_y_indi_df['operating_revenue']/pre_y_indi_df['operating_revenue'] - 1 # yoy_or 成长 营业收入同比增速
        yoy_op_df = curr_y_indi_df['operating_profit']/pre_y_indi_df['operating_profit'] - 1   # yoy_op 成长 经营利润同比增速
        yoy_ocf_df = curr_y_indi_df['ocf']/pre_y_indi_df['ocf'] - 1 # yoy_ocf           成长 经营现金流同比增速
        yoy_roe_df = (curr_y_indi_df['roe']-pre_y_indi_df['roe'])/abs(pre_y_indi_df['roe']) - 1 # yoy_roe           成长 (当期 ROE-去年同期 ROE)/去年同期 ROE 绝对值
    
    # Save File
        res_df = pd.DataFrame()
        res_df['yoy_tot_equity'] = yoy_tot_equity_df # yoy_tot_equity    成长 股东权益同比增速  (当期股东权益-去年同期股东权益)/去年同期股东权益绝对值
        res_df['yoy_total_assets'] = yoy_total_assets_df # yoy_total_assets  成长 总资产同比增速    (当期总资产-去年同期总资产)/去年同期总资产绝对值

        res_df['yoy_bps'] = yoy_bps_df # yoy_bps 成长 每股帐面价值同比增长率 (当期 BPS-去年同期 BPS)/去年同期 BPS 绝对值,其中 BPS 表示每股帐面价值

        res_df['yoy_orps'] = yoy_orps_df      # yoy_orps          成长 每股营业收入同比增长率
        res_df['yoy_eps'] = yoy_eps_df        # yoy_eps           成长 每股收益同比增速
        res_df['yoy_orps_q'] = yoy_orps_q_df  # yoy_orps_q        成长 单季度每股营业收入同比增长率
        res_df['yoy_eps_q'] = yoy_eps_q_df    # yoy_eps_q         成长 单季度每股收益同比增速

        res_df['yoy_ocfps'] = yoy_ocfps_df # yoy_ocfps         成长 每股经营现金流同比增速
        res_df['yoy_ocfps_q'] = yoy_ocfps_q_df # yoy_ocfps_q       成长 单季度每股经营现金流同比增速

        res_df['yoy_np'] = yoy_np_df # yoy_np            成长 净利润同比增速
        res_df['yoy_np_q'] = yoy_np_q_df # yoy_np_q          成长 单季度净利润同比增速

        res_df['yoy_or'] = yoy_or_df # yoy_or            成长 营业收入同比增速
        res_df['yoy_or_q'] = yoy_or_q_df # yoy_or_q          成长 单季度营业收入同比增速

        res_df['yoy_op'] = yoy_op_df # yoy_op            成长 经营利润同比增速
        res_df['yoy_op_q'] = yoy_op_q_df # yoy_op_q          成长 单季度经营利润同比增速

        res_df['yoy_ocf'] = yoy_ocf_df # yoy_ocf           成长 经营现金流同比增速
        res_df['yoy_ocf_q'] = yoy_ocf_q_df # yoy_ocf_q         成长 单季度经营现金流同比增速

        res_df['yoy_roe'] = yoy_roe_df # yoy_roe           成长 (当期 ROE-去年同期 ROE)/去年同期 ROE 绝对值
        
        res_df.to_csv('data/2_Growth/{}.csv'.format(str(current_dt)))
        

==== Yesterday=2015-01-09, Today=2015-01-12 =====
curr_q_indi_df:(1404, 19), pre_q_indi_df:(1404, 19)
curr_y_indi_df:(2015, 21), pre_y_indi_df:(2015, 21)
==== Yesterday=2015-01-16, Today=2015-01-19 =====
curr_q_indi_df:(1402, 19), pre_q_indi_df:(1402, 19)
curr_y_indi_df:(2006, 21), pre_y_indi_df:(2006, 21)
==== Yesterday=2015-01-23, Today=2015-01-26 =====
curr_q_indi_df:(1400, 19), pre_q_indi_df:(1400, 19)
curr_y_indi_df:(2002, 21), pre_y_indi_df:(2002, 21)
==== Yesterday=2015-01-30, Today=2015-02-02 =====
curr_q_indi_df:(1400, 19), pre_q_indi_df:(1400, 19)
curr_y_indi_df:(2004, 21), pre_y_indi_df:(2004, 21)
==== Yesterday=2015-02-06, Today=2015-02-09 =====
curr_q_indi_df:(1405, 19), pre_q_indi_df:(1405, 19)
curr_y_indi_df:(2006, 21), pre_y_indi_df:(2006, 21)
==== Yesterday=2015-02-13, Today=2015-02-16 =====
curr_q_indi_df:(1408, 19), pre_q_indi_df:(1408, 19)
curr_y_indi_df:(2019, 21), pre_y_indi_df:(2019, 21)
==== Yesterday=2015-02-17, Today=2015-02-25 =====
curr_q_indi_df:(1402, 19),

curr_q_indi_df:(1426, 19), pre_q_indi_df:(1426, 19)
curr_y_indi_df:(2116, 21), pre_y_indi_df:(2116, 21)
==== Yesterday=2016-01-22, Today=2016-01-25 =====
curr_q_indi_df:(1430, 19), pre_q_indi_df:(1430, 19)
curr_y_indi_df:(2133, 21), pre_y_indi_df:(2133, 21)
==== Yesterday=2016-01-29, Today=2016-02-01 =====
curr_q_indi_df:(1426, 19), pre_q_indi_df:(1426, 19)
curr_y_indi_df:(2124, 21), pre_y_indi_df:(2124, 21)
==== Yesterday=2016-02-05, Today=2016-02-15 =====
curr_q_indi_df:(1428, 19), pre_q_indi_df:(1428, 19)
curr_y_indi_df:(2122, 21), pre_y_indi_df:(2122, 21)
==== Yesterday=2016-02-19, Today=2016-02-22 =====
curr_q_indi_df:(1426, 19), pre_q_indi_df:(1426, 19)
curr_y_indi_df:(2114, 21), pre_y_indi_df:(2114, 21)
==== Yesterday=2016-02-26, Today=2016-02-29 =====
curr_q_indi_df:(1422, 19), pre_q_indi_df:(1422, 19)
curr_y_indi_df:(2113, 21), pre_y_indi_df:(2113, 21)
==== Yesterday=2016-03-04, Today=2016-03-07 =====
curr_q_indi_df:(1416, 19), pre_q_indi_df:(1416, 19)
curr_y_indi_df:(2113, 21

curr_y_indi_df:(2356, 21), pre_y_indi_df:(2356, 21)
==== Yesterday=2017-02-10, Today=2017-02-13 =====
curr_q_indi_df:(1473, 19), pre_q_indi_df:(1473, 19)
curr_y_indi_df:(2353, 21), pre_y_indi_df:(2353, 21)
==== Yesterday=2017-02-17, Today=2017-02-20 =====
curr_q_indi_df:(1471, 19), pre_q_indi_df:(1471, 19)
curr_y_indi_df:(2342, 21), pre_y_indi_df:(2342, 21)
==== Yesterday=2017-02-24, Today=2017-02-27 =====
curr_q_indi_df:(1468, 19), pre_q_indi_df:(1468, 19)
curr_y_indi_df:(2344, 21), pre_y_indi_df:(2344, 21)
==== Yesterday=2017-03-03, Today=2017-03-06 =====
curr_q_indi_df:(1460, 19), pre_q_indi_df:(1460, 19)
curr_y_indi_df:(2346, 21), pre_y_indi_df:(2346, 21)
==== Yesterday=2017-03-10, Today=2017-03-13 =====
curr_q_indi_df:(1450, 19), pre_q_indi_df:(1450, 19)
curr_y_indi_df:(2345, 21), pre_y_indi_df:(2345, 21)
==== Yesterday=2017-03-17, Today=2017-03-20 =====
curr_q_indi_df:(1435, 19), pre_q_indi_df:(1435, 19)
curr_y_indi_df:(2343, 21), pre_y_indi_df:(2343, 21)
==== Yesterday=2017-03-2

curr_q_indi_df:(1524, 19), pre_q_indi_df:(1524, 19)
curr_y_indi_df:(2542, 21), pre_y_indi_df:(2542, 21)
==== Yesterday=2018-03-02, Today=2018-03-05 =====
curr_q_indi_df:(1524, 19), pre_q_indi_df:(1524, 19)
curr_y_indi_df:(2550, 21), pre_y_indi_df:(2550, 21)
==== Yesterday=2018-03-09, Today=2018-03-12 =====
curr_q_indi_df:(1520, 19), pre_q_indi_df:(1520, 19)
curr_y_indi_df:(2560, 21), pre_y_indi_df:(2560, 21)
==== Yesterday=2018-03-16, Today=2018-03-19 =====
curr_q_indi_df:(1511, 19), pre_q_indi_df:(1511, 19)
curr_y_indi_df:(2572, 21), pre_y_indi_df:(2572, 21)
==== Yesterday=2018-03-23, Today=2018-03-26 =====
curr_q_indi_df:(1500, 19), pre_q_indi_df:(1500, 19)
curr_y_indi_df:(2574, 21), pre_y_indi_df:(2574, 21)
==== Yesterday=2018-03-30, Today=2018-04-02 =====
curr_q_indi_df:(1462, 19), pre_q_indi_df:(1462, 19)
curr_y_indi_df:(2587, 21), pre_y_indi_df:(2587, 21)
==== Yesterday=2018-04-04, Today=2018-04-09 =====
curr_q_indi_df:(1951, 19), pre_q_indi_df:(1951, 19)
curr_y_indi_df:(2607, 21

curr_y_indi_df:(3151, 21), pre_y_indi_df:(3151, 21)
==== Yesterday=2019-03-22, Today=2019-03-25 =====
curr_q_indi_df:(1600, 19), pre_q_indi_df:(1600, 19)
curr_y_indi_df:(3140, 21), pre_y_indi_df:(3140, 21)
==== Yesterday=2019-03-29, Today=2019-04-01 =====
curr_q_indi_df:(1560, 19), pre_q_indi_df:(1560, 19)
curr_y_indi_df:(3146, 21), pre_y_indi_df:(3146, 21)
==== Yesterday=2019-04-04, Today=2019-04-08 =====
curr_q_indi_df:(2138, 19), pre_q_indi_df:(2138, 19)
curr_y_indi_df:(3156, 21), pre_y_indi_df:(3156, 21)
==== Yesterday=2019-04-12, Today=2019-04-15 =====
curr_q_indi_df:(2114, 19), pre_q_indi_df:(2114, 19)
curr_y_indi_df:(3161, 21), pre_y_indi_df:(3161, 21)
==== Yesterday=2019-04-19, Today=2019-04-22 =====
curr_q_indi_df:(2058, 19), pre_q_indi_df:(2058, 19)
curr_y_indi_df:(3166, 21), pre_y_indi_df:(3166, 21)
==== Yesterday=2019-04-26, Today=2019-04-29 =====
curr_q_indi_df:(1784, 19), pre_q_indi_df:(1784, 19)
curr_y_indi_df:(3015, 21), pre_y_indi_df:(3015, 21)
==== Yesterday=2019-04-3

curr_q_indi_df:(1619, 19), pre_q_indi_df:(1619, 19)
curr_y_indi_df:(3256, 21), pre_y_indi_df:(3256, 21)
==== Yesterday=2020-04-03, Today=2020-04-07 =====
curr_q_indi_df:(2219, 19), pre_q_indi_df:(2219, 19)
curr_y_indi_df:(3254, 21), pre_y_indi_df:(3254, 21)
==== Yesterday=2020-04-10, Today=2020-04-13 =====
curr_q_indi_df:(2197, 19), pre_q_indi_df:(2197, 19)
curr_y_indi_df:(3254, 21), pre_y_indi_df:(3254, 21)
==== Yesterday=2020-04-17, Today=2020-04-20 =====
curr_q_indi_df:(2157, 19), pre_q_indi_df:(2157, 19)
curr_y_indi_df:(3258, 21), pre_y_indi_df:(3258, 21)
==== Yesterday=2020-04-24, Today=2020-04-27 =====
curr_q_indi_df:(2029, 19), pre_q_indi_df:(2029, 19)
curr_y_indi_df:(3254, 21), pre_y_indi_df:(3254, 21)
==== Yesterday=2020-04-30, Today=2020-05-06 =====
curr_q_indi_df:(1643, 19), pre_q_indi_df:(1643, 19)
curr_y_indi_df:(2792, 21), pre_y_indi_df:(2792, 21)
==== Yesterday=2020-05-08, Today=2020-05-11 =====
curr_q_indi_df:(1006, 19), pre_q_indi_df:(1006, 19)
curr_y_indi_df:(2532, 21

curr_y_indi_df:(3355, 21), pre_y_indi_df:(3355, 21)
==== Yesterday=2021-04-09, Today=2021-04-12 =====
curr_q_indi_df:(2215, 19), pre_q_indi_df:(2215, 19)
curr_y_indi_df:(3358, 21), pre_y_indi_df:(3358, 21)
==== Yesterday=2021-04-16, Today=2021-04-19 =====
curr_q_indi_df:(2169, 19), pre_q_indi_df:(2169, 19)
curr_y_indi_df:(3366, 21), pre_y_indi_df:(3366, 21)
==== Yesterday=2021-04-23, Today=2021-04-26 =====
curr_q_indi_df:(2057, 19), pre_q_indi_df:(2057, 19)
curr_y_indi_df:(3283, 21), pre_y_indi_df:(3283, 21)
==== Yesterday=2021-04-30, Today=2021-05-06 =====
curr_q_indi_df:(1676, 19), pre_q_indi_df:(1676, 19)
curr_y_indi_df:(2820, 21), pre_y_indi_df:(2820, 21)
==== Yesterday=2021-05-07, Today=2021-05-10 =====
curr_q_indi_df:(1000, 19), pre_q_indi_df:(1000, 19)
curr_y_indi_df:(2507, 21), pre_y_indi_df:(2507, 21)
==== Yesterday=2021-05-14, Today=2021-05-17 =====
curr_q_indi_df:(1000, 19), pre_q_indi_df:(1000, 19)
curr_y_indi_df:(2508, 21), pre_y_indi_df:(2508, 21)
==== Yesterday=2021-05-2

curr_q_indi_df:(2219, 19), pre_q_indi_df:(2219, 19)
curr_y_indi_df:(3513, 21), pre_y_indi_df:(3513, 21)
==== Yesterday=2022-04-29, Today=2022-05-05 =====
curr_q_indi_df:(1838, 19), pre_q_indi_df:(1838, 19)
curr_y_indi_df:(3044, 21), pre_y_indi_df:(3044, 21)
==== Yesterday=2022-05-06, Today=2022-05-09 =====
curr_q_indi_df:(1000, 19), pre_q_indi_df:(1000, 19)
curr_y_indi_df:(2502, 21), pre_y_indi_df:(2502, 21)
==== Yesterday=2022-05-13, Today=2022-05-16 =====
curr_q_indi_df:(1001, 19), pre_q_indi_df:(1001, 19)
curr_y_indi_df:(2505, 21), pre_y_indi_df:(2505, 21)
==== Yesterday=2022-05-20, Today=2022-05-23 =====
curr_q_indi_df:(1001, 19), pre_q_indi_df:(1001, 19)
curr_y_indi_df:(2506, 21), pre_y_indi_df:(2506, 21)
==== Yesterday=2022-05-27, Today=2022-05-30 =====
curr_q_indi_df:(1002, 19), pre_q_indi_df:(1002, 19)
curr_y_indi_df:(2509, 21), pre_y_indi_df:(2509, 21)
==== Yesterday=2022-06-02, Today=2022-06-06 =====
curr_q_indi_df:(1002, 19), pre_q_indi_df:(1002, 19)
curr_y_indi_df:(2509, 21

curr_y_indi_df:(2528, 21), pre_y_indi_df:(2528, 21)
==== Yesterday=2023-05-12, Today=2023-05-15 =====
curr_q_indi_df:(1001, 19), pre_q_indi_df:(1001, 19)
curr_y_indi_df:(2530, 21), pre_y_indi_df:(2530, 21)
==== Yesterday=2023-05-19, Today=2023-05-22 =====
curr_q_indi_df:(1001, 19), pre_q_indi_df:(1001, 19)
curr_y_indi_df:(2530, 21), pre_y_indi_df:(2530, 21)
==== Yesterday=2023-05-26, Today=2023-05-29 =====
curr_q_indi_df:(1001, 19), pre_q_indi_df:(1001, 19)
curr_y_indi_df:(2530, 21), pre_y_indi_df:(2530, 21)
==== Yesterday=2023-06-02, Today=2023-06-05 =====
curr_q_indi_df:(1001, 19), pre_q_indi_df:(1001, 19)
curr_y_indi_df:(2532, 21), pre_y_indi_df:(2532, 21)
==== Yesterday=2023-06-09, Today=2023-06-12 =====
curr_q_indi_df:(1001, 19), pre_q_indi_df:(1001, 19)
curr_y_indi_df:(2533, 21), pre_y_indi_df:(2533, 21)
==== Yesterday=2023-06-16, Today=2023-06-19 =====
curr_q_indi_df:(1001, 19), pre_q_indi_df:(1001, 19)
curr_y_indi_df:(2535, 21), pre_y_indi_df:(2535, 21)
==== Yesterday=2023-06-2

In [66]:
res_df

,yoy_tot_equity,yoy_total_assets,yoy_bps,yoy_orps,yoy_eps,yoy_orps_q,yoy_eps_q,yoy_ocfps,yoy_ocfps_q,yoy_np,yoy_np_q,yoy_or,yoy_or_q,yoy_op,yoy_op_q,yoy_ocf,yoy_ocf_q,yoy_roe
code,,,,,,,,,,,,,,,,,,
000001.XSHE,NaN,NaN,NaN,NaN,NaN,-0.172435,-0.131667,NaN,-0.328069,NaN,0.041975,NaN,0.051459,NaN,0.041176,NaN,-0.146281,NaN
000002.XSHE,NaN,NaN,NaN,NaN,NaN,0.240189,0.077672,NaN,-0.067395,NaN,-0.006231,NaN,0.454687,NaN,0.093904,NaN,0.093904,NaN
000004.XSHE,NaN,NaN,NaN,NaN,NaN,1.234577,0.753425,NaN,-0.122696,NaN,-0.052138,NaN,1.296634,NaN,-0.098332,NaN,-0.098332,NaN
000006.XSHE,NaN,NaN,NaN,NaN,NaN,-0.251947,-0.055024,NaN,-0.009375,NaN,-0.030483,NaN,-0.232371,NaN,0.016549,NaN,0.016549,NaN
000009.XSHE,NaN,NaN,NaN,NaN,NaN,0.051835,-0.326582,NaN,-0.329999,NaN,-0.240441,NaN,0.296020,NaN,-0.174457,NaN,-0.174457,NaN
000010.XSHE,NaN,NaN,NaN,NaN,NaN,-0.565346,10.076923,NaN,1.129197,NaN,15.715208,NaN,0.736803,NaN,7.507911,NaN,7.507911,NaN
000011.XSHE,NaN,NaN,NaN,NaN,NaN,-0.167550,-1.189555,NaN,-1.075473,NaN,-1.189107,NaN,-0.135200,NaN,-1.078406,NaN,-1.078406,NaN
000012.XSHE,NaN,NaN,NaN,NaN,NaN,0.212921,1.122689,NaN,2.326886,NaN,0.991556,NaN,0.277443,NaN,2.503862,NaN,2.503862,NaN
000014.XSHE,NaN,NaN,NaN,NaN,NaN,-0.405857,-1.393211,NaN,-5.305790,NaN,-1.370818,NaN,-0.389602,NaN,-5.423592,NaN,-5.423592,NaN


## 3_反转
- 22 clo_5d_60d           反转 5 日均价/60 日均价
- 23 mom_1y               反转 1 年的收益率
- 24 reverse_1m           反转 1 个月的收益率
- 25 close_max_div_min_1m 反转 1 月最高价/1 月最低价
- 26 return_max_1m        反转 过去一个月单日最高涨幅

### clo_5d_60d 反转 5 日均价/60 日均价
### mom_1y 反转 1 年的收益率
### reverse_1m 反转 1 个月的收益率
### close_max_div_min_1m 反转 1 月最高价/1 月最低价
### return_max_1m 反转 过去一个月单日最高涨幅

In [7]:
for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1: 
        print('Yesterday={}, Today={}'.format(yesterday, current_dt))
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)
        
        # Calculate ret_std_1m 
        prices_df = get_price(feasible_security, end_date=yesterday, fields=None, count=252, frequency="daily", panel=False)
        close_df = pd.pivot_table(prices_df, index='time', columns='code', values='close')

        mom_1y_df = close_df.iloc[-1,:]/close_df.iloc[0,:] - 1       # 1 年的收益率

        clo_5d_60d_df = close_df[-5:].mean()/close_df[-60:].mean()   # 5 日均价/60 日均价
        reverse_1m_df = close_df.iloc[-1,:]/close_df.iloc[-21,:] - 1 # 1 个月的收益率

        high_df = pd.pivot_table(prices_df, index='time', columns='code', values='high')
        low_df = pd.pivot_table(prices_df, index='time', columns='code', values='low')
        close_max_div_min_1m_df = high_df[-20:].max()/low_df[-20:].min() # 1 月最高价/1 月最低价

        r_df = close_df/close_df.shift(1)-1
        return_max_1m_df = r_df[-20:].max() # 过去一个月单日最高涨幅
        
        # Save File
        res_df = pd.DataFrame()
        res_df['clo_5d_60d'] = clo_5d_60d_df  # 5日均价 / 60日均价
        res_df['mom_252d'] = mom_1y_df        # 反转 1 年的收益率
        res_df['reverse_20d'] = reverse_1m_df # 反转 1 个月的收益率
        res_df['close_max_div_min_20d'] = close_max_div_min_1m_df # 反转 1 月最高价 / 1 月最低价
        res_df['return_max_20d'] = return_max_1m_df # 过去一个月单日最高涨幅
        
        res_df.to_csv('data/3_Reverse/{}.csv'.format(str(current_dt)))


Yesterday=2015-01-09, Today=2015-01-12
Yesterday=2015-01-16, Today=2015-01-19
Yesterday=2015-01-23, Today=2015-01-26
Yesterday=2015-01-30, Today=2015-02-02
Yesterday=2015-02-06, Today=2015-02-09
Yesterday=2015-02-13, Today=2015-02-16
Yesterday=2015-02-17, Today=2015-02-25
Yesterday=2015-02-27, Today=2015-03-02
Yesterday=2015-03-06, Today=2015-03-09
Yesterday=2015-03-13, Today=2015-03-16
Yesterday=2015-03-20, Today=2015-03-23
Yesterday=2015-03-27, Today=2015-03-30
Yesterday=2015-04-03, Today=2015-04-07
Yesterday=2015-04-10, Today=2015-04-13
Yesterday=2015-04-17, Today=2015-04-20
Yesterday=2015-04-24, Today=2015-04-27
Yesterday=2015-04-30, Today=2015-05-04
Yesterday=2015-05-08, Today=2015-05-11
Yesterday=2015-05-15, Today=2015-05-18
Yesterday=2015-05-22, Today=2015-05-25
Yesterday=2015-05-29, Today=2015-06-01
Yesterday=2015-06-05, Today=2015-06-08
Yesterday=2015-06-12, Today=2015-06-15
Yesterday=2015-06-19, Today=2015-06-23
Yesterday=2015-06-26, Today=2015-06-29
Yesterday=2015-07-03, Tod

Yesterday=2019-06-06, Today=2019-06-10
Yesterday=2019-06-14, Today=2019-06-17
Yesterday=2019-06-21, Today=2019-06-24
Yesterday=2019-06-28, Today=2019-07-01
Yesterday=2019-07-05, Today=2019-07-08
Yesterday=2019-07-12, Today=2019-07-15
Yesterday=2019-07-19, Today=2019-07-22
Yesterday=2019-07-26, Today=2019-07-29
Yesterday=2019-08-02, Today=2019-08-05
Yesterday=2019-08-09, Today=2019-08-12
Yesterday=2019-08-16, Today=2019-08-19
Yesterday=2019-08-23, Today=2019-08-26
Yesterday=2019-08-30, Today=2019-09-02
Yesterday=2019-09-06, Today=2019-09-09
Yesterday=2019-09-12, Today=2019-09-16
Yesterday=2019-09-20, Today=2019-09-23
Yesterday=2019-09-27, Today=2019-09-30
Yesterday=2019-09-30, Today=2019-10-08
Yesterday=2019-10-11, Today=2019-10-14
Yesterday=2019-10-18, Today=2019-10-21
Yesterday=2019-10-25, Today=2019-10-28
Yesterday=2019-11-01, Today=2019-11-04
Yesterday=2019-11-08, Today=2019-11-11
Yesterday=2019-11-15, Today=2019-11-18
Yesterday=2019-11-22, Today=2019-11-25
Yesterday=2019-11-29, Tod

Yesterday=2023-07-07, Today=2023-07-10


## 4_杠杆
- 27 current_ratio          杠杆 流动资产/流动负债
- 28 debt_asset_ratio       杠杆 资产/负债
- 29 current_liab_ratio     杠杆 流动负债/总资产

### current_ratio 杠杆 流动资产/流动负债
### debt_asset_ratio 杠杆 资产/负债
### current_liab_ratio 杠杆 流动负债/总资产

In [4]:
for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1: 
        print('Yesterday={}, Today={}'.format(yesterday, current_dt))
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)
            
    # Calculate Indicators
        q = query(
                valuation.code, 
                balance.total_current_assets,    # 流动资产
                balance.total_current_liability, # 流动负债
                balance.total_assets,    # 总资产
                balance.total_liability, # 总负债
                ).filter(valuation.code.in_(feasible_security))

        df = get_fundamentals(q, yesterday).set_index('code')

        current_ratio_df = df['total_current_assets']/df['total_current_liability']
        debt_asset_ratio_df = df['total_assets']/df['total_liability']
        current_liab_ratio_df = df['total_current_liability']/df['total_assets']
        
    # Save File
        res_df = pd.DataFrame()
        res_df['current_ratio'] = current_ratio_df
        res_df['debt_asset_ratio'] = debt_asset_ratio_df
        res_df['current_liab_ratio'] = current_liab_ratio_df
        res_df.to_csv('data/4_Leverage/{}.csv'.format(str(current_dt)))


Yesterday=2015-01-09, Today=2015-01-12
Yesterday=2015-01-16, Today=2015-01-19
Yesterday=2015-01-23, Today=2015-01-26
Yesterday=2015-01-30, Today=2015-02-02
Yesterday=2015-02-06, Today=2015-02-09
Yesterday=2015-02-13, Today=2015-02-16
Yesterday=2015-02-17, Today=2015-02-25
Yesterday=2015-02-27, Today=2015-03-02
Yesterday=2015-03-06, Today=2015-03-09
Yesterday=2015-03-13, Today=2015-03-16
Yesterday=2015-03-20, Today=2015-03-23
Yesterday=2015-03-27, Today=2015-03-30
Yesterday=2015-04-03, Today=2015-04-07
Yesterday=2015-04-10, Today=2015-04-13
Yesterday=2015-04-17, Today=2015-04-20
Yesterday=2015-04-24, Today=2015-04-27
Yesterday=2015-04-30, Today=2015-05-04
Yesterday=2015-05-08, Today=2015-05-11
Yesterday=2015-05-15, Today=2015-05-18
Yesterday=2015-05-22, Today=2015-05-25
Yesterday=2015-05-29, Today=2015-06-01
Yesterday=2015-06-05, Today=2015-06-08
Yesterday=2015-06-12, Today=2015-06-15
Yesterday=2015-06-19, Today=2015-06-23
Yesterday=2015-06-26, Today=2015-06-29
Yesterday=2015-07-03, Tod

Yesterday=2019-05-17, Today=2019-05-20
Yesterday=2019-05-24, Today=2019-05-27
Yesterday=2019-05-31, Today=2019-06-03
Yesterday=2019-06-06, Today=2019-06-10
Yesterday=2019-06-14, Today=2019-06-17
Yesterday=2019-06-21, Today=2019-06-24
Yesterday=2019-06-28, Today=2019-07-01
Yesterday=2019-07-05, Today=2019-07-08
Yesterday=2019-07-12, Today=2019-07-15
Yesterday=2019-07-19, Today=2019-07-22
Yesterday=2019-07-26, Today=2019-07-29
Yesterday=2019-08-02, Today=2019-08-05
Yesterday=2019-08-09, Today=2019-08-12
Yesterday=2019-08-16, Today=2019-08-19
Yesterday=2019-08-23, Today=2019-08-26
Yesterday=2019-08-30, Today=2019-09-02
Yesterday=2019-09-06, Today=2019-09-09
Yesterday=2019-09-12, Today=2019-09-16
Yesterday=2019-09-20, Today=2019-09-23
Yesterday=2019-09-27, Today=2019-09-30
Yesterday=2019-09-30, Today=2019-10-08
Yesterday=2019-10-11, Today=2019-10-14
Yesterday=2019-10-18, Today=2019-10-21
Yesterday=2019-10-25, Today=2019-10-28
Yesterday=2019-11-01, Today=2019-11-04
Yesterday=2019-11-08, Tod

Yesterday=2023-06-16, Today=2023-06-19
Yesterday=2023-06-21, Today=2023-06-26
Yesterday=2023-06-30, Today=2023-07-03
Yesterday=2023-07-07, Today=2023-07-10


## 5_红利
- 30 dividend_yield_ratio   红利 股息率

### 30 dividend_yield_ratio 红利 股息率

In [ ]:
for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1: 
        print('Yesterday={}, Today={}'.format(yesterday, current_dt))
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)
            
    # Calculate Indicators
        q = query(
            valuation.code, 
            balance.dividend_payable,    # 应付股息
            valuation.market_cap
            ).filter(valuation.code.in_(feasible_security))

        df = get_fundamentals(q, yesterday).set_index('code')
        df['dividend_yield_ratio'] = df['dividend_payable'].fillna(0)/df['market_cap']

    # Save File
        res_df = df[['dividend_yield_ratio']]
        res_df.to_csv('data/5_Dividend/{}.csv'.format(str(current_dt)))


Yesterday=2015-01-09, Today=2015-01-12
Yesterday=2015-01-16, Today=2015-01-19
Yesterday=2015-01-23, Today=2015-01-26
Yesterday=2015-01-30, Today=2015-02-02
Yesterday=2015-02-06, Today=2015-02-09
Yesterday=2015-02-13, Today=2015-02-16
Yesterday=2015-02-17, Today=2015-02-25
Yesterday=2015-02-27, Today=2015-03-02
Yesterday=2015-03-06, Today=2015-03-09
Yesterday=2015-03-13, Today=2015-03-16
Yesterday=2015-03-20, Today=2015-03-23
Yesterday=2015-03-27, Today=2015-03-30
Yesterday=2015-04-03, Today=2015-04-07
Yesterday=2015-04-10, Today=2015-04-13
Yesterday=2015-04-17, Today=2015-04-20
Yesterday=2015-04-24, Today=2015-04-27
Yesterday=2015-04-30, Today=2015-05-04
Yesterday=2015-05-08, Today=2015-05-11
Yesterday=2015-05-15, Today=2015-05-18
Yesterday=2015-05-22, Today=2015-05-25
Yesterday=2015-05-29, Today=2015-06-01
Yesterday=2015-06-05, Today=2015-06-08
Yesterday=2015-06-12, Today=2015-06-15
Yesterday=2015-06-19, Today=2015-06-23
Yesterday=2015-06-26, Today=2015-06-29
Yesterday=2015-07-03, Tod

Yesterday=2019-03-01, Today=2019-03-04
Yesterday=2019-03-08, Today=2019-03-11
Yesterday=2019-03-15, Today=2019-03-18
Yesterday=2019-03-22, Today=2019-03-25
Yesterday=2019-03-29, Today=2019-04-01
Yesterday=2019-04-04, Today=2019-04-08
Yesterday=2019-04-12, Today=2019-04-15
Yesterday=2019-04-19, Today=2019-04-22
Yesterday=2019-04-26, Today=2019-04-29
Yesterday=2019-04-30, Today=2019-05-06
Yesterday=2019-05-10, Today=2019-05-13
Yesterday=2019-05-17, Today=2019-05-20
Yesterday=2019-05-24, Today=2019-05-27
Yesterday=2019-05-31, Today=2019-06-03
Yesterday=2019-06-06, Today=2019-06-10
Yesterday=2019-06-14, Today=2019-06-17
Yesterday=2019-06-21, Today=2019-06-24
Yesterday=2019-06-28, Today=2019-07-01
Yesterday=2019-07-05, Today=2019-07-08
Yesterday=2019-07-12, Today=2019-07-15
Yesterday=2019-07-19, Today=2019-07-22
Yesterday=2019-07-26, Today=2019-07-29
Yesterday=2019-08-02, Today=2019-08-05
Yesterday=2019-08-09, Today=2019-08-12
Yesterday=2019-08-16, Today=2019-08-19
Yesterday=2019-08-23, Tod

## 6_价值
- 31 bp             价值 股东权益(报告期)/总市值
- 32 ep             价值 市盈率倒数
- 33 cfp            价值 市现率倒数
- 34 sp             价值 市销率倒数
- 35 fcffp          价值 企业自由现金流 ttm/总市值
- 36 ocfp           价值 经营现金流_TTM/总市值
- 37 value_residual 价值 总市值的对数对账面价值对数，净利润对数以及负债合计对数 做横截面回归的取残差

### bp 价值 股东权益(报告期)/总市值
### ep 价值 市盈率倒数
### cfp 价值 市现率倒数
### sp 价值 市销率倒数
### fcffp 价值 企业自由现金流 ttm/总市值
### ocfp 价值 经营现金流_TTM/总市值
### value_residual 价值 总市值的对数对账面价值对数，净利润对数以及负债合计对数 做横截面回归的取残差

In [ ]:
from jqdata import get_history_fundamentals
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()


for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1: 
        print('Yesterday={}, Today={}'.format(yesterday, current_dt))
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)
        
    # Calculate Indicators
        q = query(
            valuation.code, 
            valuation.market_cap,             # 总市值
            valuation.circulating_market_cap, # 流通市值
            valuation.pe_ratio,               # 市盈率(PE, TTM)
            valuation.pcf_ratio,              # 市现率(PCF, 现金净流量TTM)
            valuation.ps_ratio,               # 市销率(PS, TTM)
            
            balance.total_assets,        # 总资产
            balance.total_liability,     # 总负债
            balance.total_owner_equities,# 总所有者权益
            
            income.total_operating_revenue,# 营业总收入(元) 具体核算范围和方法参见上市公司定期报告
            income.operating_revenue,      # 营业收入(元)   具体核算范围和方法参见上市公司定期报告

            income.total_operating_cost,# 营业总成本(元) 营业总成本=主营业务成本+其他业务成本+利息支出+手续费及佣金支出+退保金+赔付支出净额+提取保险合同准备金净额+保单红利支出+分保费用+营业税金及附加+销售费用+管理费用+财务费用+资产减值损失+其他
            income.operating_cost,      # 营业成本(元)   营业成本，也称运营成本。是指企业所销售商品或者提供劳务的成本。营业成本应当与所销售商品或者所提供劳务而取得的收入进行配比。
            income.operating_profit,    # 营业利润(元)   营业利润是企业最基本经营活动的成果，也是企业一定时期获得利润中最主要、最稳定的来源。2006年财政部颁布的新企业会计准则-30号财务报表列报中已对营业利润进行了调整，将投资收益调入营业利润，同时取消了主营业务利润和其他业务利润的提法，补贴收入被并入营业外收入，营业利润减营业外收支调整即得到利润总额。

            income.total_profit,        # 利润总额(元)   利润总额指企业在生产经营过程中各种收入扣除各种耗费后的盈余，反映企业在报告期内实现的盈亏总额。
            income.income_tax_expense,  # 所得税费用(元) 所得税费用是指企业经营利润应交纳的所得税。“所得税费用”，核算企业负担的所得税，是损益类科目；这一般不等于当期应交所得税，因为可能存在“暂时性差异”。如果只有永久性差异，则等于当期应交所得税。
            income.net_profit,          # 净利润                

            income.interest_income,     # 利息收入(元) 利息收入是指纳税人购买各种债券等有价证券的利息，外单位欠款付给的利息以及其他利息收入。包括：购买各种债券等有价证券的利息，如购买国库券，重点企业建设债券、国家保值公债以及政府部门和企业发放的各类有价证券；企业各项存款所取得的利息外单位欠本企业款而取得的利息；其他利息收入等。
            income.interest_expense,    # 利息支出(元) 利息支出是指临时借款的利息支出。在以收付实现制作为记帐基础的前提条件下，所谓支出应以实际支付为标准，即资金流出，标志着现金、银行存款的减少。就利息支出而言、给个人帐户计息，其资金并没有流出，现金、银行存款并没有减少，因此，给个人计息不应作为利息支出列支。
            
            cash_flow.net_operate_cash_flow,
            
            (balance.total_owner_equities/valuation.market_cap/100000000.0).label("BP"), # 股东权益 / 总市值
            ).filter(valuation.code.in_(feasible_security))
        df = get_fundamentals(q, yesterday).set_index('code')
        
        # NCA_inc
        fields = [
            balance.total_non_current_assets,
        ]
        q_df = get_history_fundamentals(feasible_security, fields, watch_date=yesterday, count=5, interval='1q', stat_by_year=False)
        min_q_dates = q_df.groupby('code')['statDate'].min()
        pre_q_indi_df = pd.merge(q_df, min_q_dates.reset_index(), how='right', on=['code','statDate']).set_index('code')
        max_q_dates = q_df.groupby('code')['statDate'].max()
        curr_q_indi_df = pd.merge(q_df, max_q_dates.reset_index(), how='right', on=['code','statDate']).set_index('code')
        # fcff
        df['NCA_inc'] = curr_q_indi_df['total_non_current_assets'] - pre_q_indi_df['total_non_current_assets']
        df['fcff'] = (df['net_operate_cash_flow'].fillna(0) + 
                      df['interest_expense'].fillna(0)*0.75 - 
                      df['NCA_inc'].fillna(0))
        # 37 value_residual 价值 [总市值的对数]vs.[账面价值对数，净利润对数以及负债合计对数]做横截面回归的取残差
        log_df = log(df[['total_assets', 'total_liability', 'net_profit', 'market_cap']]).dropna()
        lr_model.fit(log_df[['total_assets', 'total_liability', 'net_profit']], log_df['market_cap'])
        log_df['value_residual'] = log_df['market_cap'] - lr_model.predict(log_df[['total_assets', 'total_liability', 'net_profit']])
    # Save File 
        res_df = pd.DataFrame()
        res_df['market_cap'] = df['market_cap']
        res_df['circulating_market_cap'] = df['circulating_market_cap']
        res_df['bp'] = df['BP'] # 31 bp 价值 股东权益(报告期)/总市值
        res_df['ep'] = 1/df['pe_ratio'] # 32 ep 价值 市盈率倒数
        res_df['cfp'] = 1/df['pcf_ratio'] # 33 cfp 价值 市现率倒数
        res_df['sp'] = 1/df['ps_ratio']  # 34 sp 价值 市销率倒数
        res_df['fcffp'] = df['fcff']/df['market_cap'] # 35 fcffp 价值 企业自由现金流 ttm/总市值
        res_df['ocfp'] = (df['operating_profit']+df['interest_expense'])/df['market_cap'] # 36 ocfp 价值 经营现金流_TTM/总市值
        res_df['value_residual'] = log_df['value_residual'] # 37 value_residual 价值 [总市值的对数]vs.[账面价值对数，净利润对数以及负债合计对数]做横截面回归的取残差

        res_df.to_csv('data/6_Value/{}.csv'.format(str(current_dt)))


Yesterday=2015-01-09, Today=2015-01-12


/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:61: RuntimeWarning: invalid value encountered in log


Yesterday=2015-01-16, Today=2015-01-19


/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:61: RuntimeWarning: invalid value encountered in log


Yesterday=2015-01-23, Today=2015-01-26


...

Yesterday=2016-11-11, Today=2016-11-14


/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:61: RuntimeWarning: invalid value encountered in log


Yesterday=2016-11-18, Today=2016-11-21


/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:61: RuntimeWarning: invalid value encountered in log


Yesterday=2016-11-25, Today=2016-11-28


## 7_流动性
- 38 amt_1m_3m           流动性 过去 1 个月日均成交额/过去三个月日均成交额
- 39 corr_close_turnover 流动性 收盘价与换手率的相关系数
- 40 illiq               流动性 每天一个亿成交量能推动的股价涨幅
- 41 ln_volume_mean_1m   流动性 成交量对数一个月均值
- 42 turnover_mean_1m    流动性 换手率一个月均值

### amt_1m_3m 流动性 过去 1 个月日均成交额/过去三个月日均成交额
### corr_close_turnover 流动性 收盘价与换手率的相关系数
### illiq 流动性 每天一个亿成交量能推动的股价涨幅
### ln_volume_mean_1m 流动性 成交量对数一个月均值
### turnover_mean_1m 流动性 换手率一个月均值

In [4]:
import gc

for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1 and str(current_dt)>'2021-06-15': 
        print('Yesterday={}, Today={}'.format(yesterday, current_dt))
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)
        size = len(feasible_security)
        
    # Calculate Indicators
        prices_df = get_price(feasible_security, end_date=yesterday, fields=None, count=60, \
                              frequency="daily", panel=False)
        close_df = pd.pivot_table(prices_df, index='time', columns='code', values='close') # 收盘价
        volume_df = pd.pivot_table(prices_df, index='time', columns='code', values='volume') # 成交量
        r_df = close_df/close_df.shift(1)-1
        
        amt_1m_3m_df = volume_df[-20:].mean()/volume_df[-60:].mean()        # 38 amt_1m_3m 流动性 过去 1 个月日均成交额/过去三个月日均成交额
        illiq_df =  (1000000000 * r_df[-5:].mean() / volume_df[-5:].mean()) # 40 illiq 流动性 每天一个亿成交量能推动的股价涨幅
        ln_volume_mean_1m_df = log(volume_df[-20:]).mean()                  # 41 ln_volume_mean_1m 流动性 成交量对数一个月均值
        
        res_df = pd.DataFrame()
        res_df['amt_1m_3m'] = amt_1m_3m_df
        res_df['illiq'] = illiq_df
        res_df['ln_volume_mean_1m'] = ln_volume_mean_1m_df
        del amt_1m_3m_df, illiq_df, ln_volume_mean_1m_df
        gc.collect()
        
        close_df = close_df[-20:] # 后边只会用到20d的close_df
        turnover_ratio_df = close_df.copy() * np.nan # 换手率
        q = query(valuation.code, 
                  valuation.turnover_ratio, # 换手率
                 ).filter(valuation.code.in_(feasible_security))
        for d in close_df.index:
            turnover_ratio_df.loc[d,:] = get_fundamentals(q, d)['turnover_ratio'].values # 换手率        
        
        corr_arr = np.corrcoef(turnover_ratio_df, close_df, rowvar=False)
        corr_df = pd.Series(np.diagonal(corr_arr[size:, :size]), index=turnover_ratio_df.columns)
        
        corr_close_turnover_df = corr_df               # 39 corr_close_turnover 流动性 收盘价与换手率的相关系数
        turnover_mean_1m_df = turnover_ratio_df.mean() # 42 turnover_mean_1m    流动性 换手率一个月均值
        
        res_df['corr_close_turnover'] = corr_close_turnover_df
        res_df['turnover_mean_1m'] = turnover_mean_1m_df        
        del corr_arr, corr_df, corr_close_turnover_df, turnover_mean_1m_df
        gc.collect()
    # Save File         
        res_df.to_csv('data/7_Liquidity/{}.csv'.format(str(current_dt)))


Yesterday=2021-06-18, Today=2021-06-21


/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:20: RuntimeWarning: divide by zero encountered in log


Yesterday=2021-06-25, Today=2021-06-28


/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:20: RuntimeWarning: divide by zero encountered in log


Yesterday=2021-07-02, Today=2021-07-05


...

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:20: RuntimeWarning: divide by zero encountered in log


Yesterday=2023-06-30, Today=2023-07-03


/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:20: RuntimeWarning: divide by zero encountered in log


Yesterday=2023-07-07, Today=2023-07-10


/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:20: RuntimeWarning: divide by zero encountered in log


## 8_盈利
- 43 roe                盈利 roe_ttm
- 44 eps_adjust         盈利 扣除非经常损益后的净利润/总股本
- 45 grossmargin_ratio  盈利 毛利_TTM/TTM 营业总收入
- 46 grossprofit_assets 盈利 毛利_TTM/总资产
- 47 grossprofit        盈利 (营业收入-营业支出)/营业收入
- 48 roa                盈利 roa2_ttm
- 49 roic               盈利 roic_ttm

### roe 盈利 roe_ttm
### eps_adjust 盈利 扣除非经常损益后的净利润/总股本
### grossmargin_ratio 盈利 毛利_TTM/TTM 营业总收入
### grossprofit_assets 盈利 毛利_TTM/总资产
### grossprofit 盈利 (营业收入-营业支出)/营业收入
### roa 盈利 roa2_ttm
### roic 盈利 roic_ttm

In [10]:
for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1: 
        print('Yesterday={}, Today={}'.format(yesterday, current_dt))
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)
        
    # Calculate Indicators
        q = query(
            valuation.code, # 股票代码
            
            income.total_operating_revenue,# 营业总收入(元) 具体核算范围和方法参见上市公司定期报告
            income.operating_revenue,      # 营业收入(元)   具体核算范围和方法参见上市公司定期报告
            
            income.total_operating_cost,# 营业总成本(元) 营业总成本=主营业务成本+其他业务成本+利息支出+手续费及佣金支出+退保金+赔付支出净额+提取保险合同准备金净额+保单红利支出+分保费用+营业税金及附加+销售费用+管理费用+财务费用+资产减值损失+其他
            income.operating_cost,      # 营业成本(元)   营业成本，也称运营成本。是指企业所销售商品或者提供劳务的成本。营业成本应当与所销售商品或者所提供劳务而取得的收入进行配比。
            income.operating_profit,    # 营业利润(元)   营业利润是企业最基本经营活动的成果，也是企业一定时期获得利润中最主要、最稳定的来源。2006年财政部颁布的新企业会计准则-30号财务报表列报中已对营业利润进行了调整，将投资收益调入营业利润，同时取消了主营业务利润和其他业务利润的提法，补贴收入被并入营业外收入，营业利润减营业外收支调整即得到利润总额。

            income.total_profit,        # 利润总额(元)   利润总额指企业在生产经营过程中各种收入扣除各种耗费后的盈余，反映企业在报告期内实现的盈亏总额。
            income.income_tax_expense,  # 所得税费用(元) 所得税费用是指企业经营利润应交纳的所得税。“所得税费用”，核算企业负担的所得税，是损益类科目；这一般不等于当期应交所得税，因为可能存在“暂时性差异”。如果只有永久性差异，则等于当期应交所得税。
            income.net_profit,          # 净利润                
            
            balance.total_assets,        # 总资产
            balance.total_liability,     # 总负债
            balance.total_owner_equities,# 总所有者权益
            
            income.interest_income,     # 利息收入(元) 利息收入是指纳税人购买各种债券等有价证券的利息，外单位欠款付给的利息以及其他利息收入。包括：购买各种债券等有价证券的利息，如购买国库券，重点企业建设债券、国家保值公债以及政府部门和企业发放的各类有价证券；企业各项存款所取得的利息外单位欠本企业款而取得的利息；其他利息收入等。
            income.interest_expense,    # 利息支出(元) 利息支出是指临时借款的利息支出。在以收付实现制作为记帐基础的前提条件下，所谓支出应以实际支付为标准，即资金流出，标志着现金、银行存款的减少。就利息支出而言、给个人帐户计息，其资金并没有流出，现金、银行存款并没有减少，因此，给个人计息不应作为利息支出列支。

#             indicator.net_profit_margin,   # 销售净利率(%) 利润/营业收入
#             indicator.gross_profit_margin, # 销售毛利率(%) 毛利/营业收入
            
            indicator.roe,                 # 净资产收益率ROE(%) => 归属于母公司股东的净利润*2/（期初归属于母公司股东的净资产+期末归属于母公司股东的净资产）
            indicator.inc_return.label('eps_adjust'),          # 净资产收益率(扣除非经常损益)(%) => 扣除非经常损益后的净利润（不含少数股东损益）*2/（期初归属于母公司股东的净资产+期末归属于母公司股东的净资产）
            indicator.gross_profit_margin.label('grossmargin_ratio'), # 销售毛利率(%) => 毛利/营业收入
            (income.operating_profit/balance.total_assets).label("grossprofit_assets"), # 毛利_TTM/总资产
            indicator.operation_profit_to_total_revenue.label('grossprofit'),# 营业利润/营业总收入(%) => 营业利润/营业总收入(%)
            indicator.roa,                              # 总资产净利率ROA(%) => 净利润*2/（期初总资产+期末总资产）
            ((income.net_profit+income.interest_expense)/ \
             (balance.total_liability+balance.total_liability)).label("roic") # ROIC => 息前税后利润/投入资本（股本+负债）
            ).filter(valuation.code.in_(feasible_security))

        df = get_fundamentals(q, yesterday).set_index('code')

    # Save File 
        df[['roe',
            'eps_adjust',
            'grossmargin_ratio',
            'grossprofit_assets',
            'grossprofit',
            'roa',
            'roic']].to_csv('data/8_Profit/{}.csv'.format(str(current_dt)))


Yesterday=2015-01-09, Today=2015-01-12
Yesterday=2015-01-16, Today=2015-01-19
Yesterday=2015-01-23, Today=2015-01-26
Yesterday=2015-01-30, Today=2015-02-02
Yesterday=2015-02-06, Today=2015-02-09
Yesterday=2015-02-13, Today=2015-02-16
Yesterday=2015-02-17, Today=2015-02-25
Yesterday=2015-02-27, Today=2015-03-02
Yesterday=2015-03-06, Today=2015-03-09
Yesterday=2015-03-13, Today=2015-03-16
Yesterday=2015-03-20, Today=2015-03-23
Yesterday=2015-03-27, Today=2015-03-30
Yesterday=2015-04-03, Today=2015-04-07
Yesterday=2015-04-10, Today=2015-04-13
Yesterday=2015-04-17, Today=2015-04-20
Yesterday=2015-04-24, Today=2015-04-27
Yesterday=2015-04-30, Today=2015-05-04
Yesterday=2015-05-08, Today=2015-05-11
Yesterday=2015-05-15, Today=2015-05-18
Yesterday=2015-05-22, Today=2015-05-25
Yesterday=2015-05-29, Today=2015-06-01
Yesterday=2015-06-05, Today=2015-06-08
Yesterday=2015-06-12, Today=2015-06-15
Yesterday=2015-06-19, Today=2015-06-23
Yesterday=2015-06-26, Today=2015-06-29
Yesterday=2015-07-03, Tod

Yesterday=2019-03-01, Today=2019-03-04
Yesterday=2019-03-08, Today=2019-03-11
Yesterday=2019-03-15, Today=2019-03-18
Yesterday=2019-03-22, Today=2019-03-25
Yesterday=2019-03-29, Today=2019-04-01
Yesterday=2019-04-04, Today=2019-04-08
Yesterday=2019-04-12, Today=2019-04-15
Yesterday=2019-04-19, Today=2019-04-22
Yesterday=2019-04-26, Today=2019-04-29
Yesterday=2019-04-30, Today=2019-05-06
Yesterday=2019-05-10, Today=2019-05-13
Yesterday=2019-05-17, Today=2019-05-20
Yesterday=2019-05-24, Today=2019-05-27
Yesterday=2019-05-31, Today=2019-06-03
Yesterday=2019-06-06, Today=2019-06-10
Yesterday=2019-06-14, Today=2019-06-17
Yesterday=2019-06-21, Today=2019-06-24
Yesterday=2019-06-28, Today=2019-07-01
Yesterday=2019-07-05, Today=2019-07-08
Yesterday=2019-07-12, Today=2019-07-15
Yesterday=2019-07-19, Today=2019-07-22
Yesterday=2019-07-26, Today=2019-07-29
Yesterday=2019-08-02, Today=2019-08-05
Yesterday=2019-08-09, Today=2019-08-12
Yesterday=2019-08-16, Today=2019-08-19
Yesterday=2019-08-23, Tod

Yesterday=2023-04-04, Today=2023-04-06
Yesterday=2023-04-07, Today=2023-04-10
Yesterday=2023-04-14, Today=2023-04-17
Yesterday=2023-04-21, Today=2023-04-24
Yesterday=2023-04-28, Today=2023-05-04
Yesterday=2023-05-05, Today=2023-05-08
Yesterday=2023-05-12, Today=2023-05-15
Yesterday=2023-05-19, Today=2023-05-22
Yesterday=2023-05-26, Today=2023-05-29
Yesterday=2023-06-02, Today=2023-06-05
Yesterday=2023-06-09, Today=2023-06-12
Yesterday=2023-06-16, Today=2023-06-19
Yesterday=2023-06-21, Today=2023-06-26
Yesterday=2023-06-30, Today=2023-07-03
Yesterday=2023-07-07, Today=2023-07-10


## 9_质量
- 50 safexp_operrev    质量 (销售费用_TTM+管理费用_TTM+财务费用_TTM)/营业收入_TTM
- 51 adm_exp_ratio     质量 管理费用_TTM/营业收入_TTM
- 52 bps               质量 (归属母公司股东的权益-其他权益工具)/总股本
- 53 currency_ratio    质量 经营活动产生的现金净流量/净利润
- 54 asset_turnover    质量 营业收入_TTM/平均总资产
- 55 inv_turnover      质量 营业成本_TTM/平均存货
- 56 fina_exp_ratio    质量 财务费用_TTM/营业收入_TTM
- 57 fix_ratio         质量 固定资产/归属母公司股东权益
- 58 networkcap_assets 质量 净运营资本/总资产(其中净营运资本=流动资产-流动负债)
- 59 moneycap_assets   质量 现金/总资产

### safexp_operrev 质量 (销售费用_TTM+管理费用_TTM+财务费用_TTM)/营业收入_TTM
### adm_exp_ratio 质量 管理费用_TTM/营业收入_TTM
### bps 质量 (归属母公司股东的权益-其他权益工具)/总股本
### currency_ratio 质量 经营活动产生的现金净流量/净利润
### asset_turnover 质量 营业收入_TTM/平均总资产
### inv_turnover 质量 营业成本_TTM/平均存货
### fina_exp_ratio 质量 财务费用_TTM/营业收入_TTM
### fix_ratio 质量 固定资产/归属母公司股东权益
### networkcap_assets 质量 净运营资本/总资产(其中净营运资本=流动资产-流动负债)
### moneycap_assets 质量 现金/总资产

In [19]:
from jqdata import get_valuation

for yesterday, current_dt in zip(trade_days[:-1], trade_days[1:]):
    if (current_dt - yesterday).days>1: 
        print('Yesterday={}, Today={}'.format(yesterday, current_dt))
        all_security_df = get_all_securities("stock", yesterday)
        all_security = all_security_df.index.tolist()
        feasible_security = filter_stocks(all_security_df, yesterday, current_dt)
        
    # Calculate Indicators
        q = query(
            valuation.code, # 股票代码
            
            income.total_operating_revenue,# 营业总收入(元) 具体核算范围和方法参见上市公司定期报告
            income.operating_revenue,      # 营业收入(元)   具体核算范围和方法参见上市公司定期报告
            
            income.total_operating_cost,   # 营业总成本(元) 营业总成本=主营业务成本+其他业务成本+利息支出+手续费及佣金支出+退保金+赔付支出净额+提取保险合同准备金净额+保单红利支出+分保费用+营业税金及附加+销售费用+管理费用+财务费用+资产减值损失+其他
            income.operating_cost,         # 营业成本(元)   营业成本，也称运营成本。是指企业所销售商品或者提供劳务的成本。营业成本应当与所销售商品或者所提供劳务而取得的收入进行配比。
            income.operating_profit,       # 营业利润(元)   营业利润是企业最基本经营活动的成果，也是企业一定时期获得利润中最主要、最稳定的来源。2006年财政部颁布的新企业会计准则-30号财务报表列报中已对营业利润进行了调整，将投资收益调入营业利润，同时取消了主营业务利润和其他业务利润的提法，补贴收入被并入营业外收入，营业利润减营业外收支调整即得到利润总额。

            income.total_profit,           # 利润总额(元)   利润总额指企业在生产经营过程中各种收入扣除各种耗费后的盈余，反映企业在报告期内实现的盈亏总额。
            income.income_tax_expense,     # 所得税费用(元) 所得税费用是指企业经营利润应交纳的所得税。“所得税费用”，核算企业负担的所得税，是损益类科目；这一般不等于当期应交所得税，因为可能存在“暂时性差异”。如果只有永久性差异，则等于当期应交所得税。
            income.net_profit,             # 净利润                
            
            income.sale_expense,           # 销售费用(元)
            income.administration_expense, # 管理费用(元)
            income.financial_expense,      # 财务费用(元)
            
            income.interest_income,        # 利息收入(元) 利息收入是指纳税人购买各种债券等有价证券的利息，外单位欠款付给的利息以及其他利息收入。包括：购买各种债券等有价证券的利息，如购买国库券，重点企业建设债券、国家保值公债以及政府部门和企业发放的各类有价证券；企业各项存款所取得的利息外单位欠本企业款而取得的利息；其他利息收入等。
            income.interest_expense,       # 利息支出(元) 利息支出是指临时借款的利息支出。在以收付实现制作为记帐基础的前提条件下，所谓支出应以实际支付为标准，即资金流出，标志着现金、银行存款的减少。就利息支出而言、给个人帐户计息，其资金并没有流出，现金、银行存款并没有减少，因此，给个人计息不应作为利息支出列支。

            balance.total_assets,          # 总资产
            balance.total_liability,       # 总负债
            balance.total_owner_equities,  # 总所有者权益
            
            balance.equities_parent_company_owners, # 归属于母公司股东权益合计(元)
            balance.total_current_assets,    # 流动资产
            balance.total_current_liability, # 流动负债
            
            balance.fixed_assets,            # 固定资产(元)
            balance.inventories,             # 存货(元)
            balance.cash_equivalents,        # 货币资金(元)
            
            cash_flow.net_operate_cash_flow, # 经营活动的净现金流
            
            ((income.sale_expense+income.administration_expense+income.financial_expense)
             /income.operating_revenue).label('safexp_operrev'), # 质量 (销售费用_TTM+管理费用_TTM+财务费用_TTM)/营业收入_TTM
            (income.administration_expense/income.operating_revenue).label('adm_exp_ratio'), # 质量 管理费用_TTM/营业收入_TTM
            # bps 质量 (归属母公司股东的权益-其他权益工具)/总股本
            (cash_flow.net_operate_cash_flow/income.net_profit).label('currency_ratio'), # 质量 经营活动产生的现金净流量/净利润
            (income.operating_revenue/balance.total_assets).label('asset_turnover'), # 质量 营业收入_TTM/平均总资产
            (income.operating_cost/balance.inventories).label('inv_turnover'), # 质量 营业成本_TTM/平均存货
            (income.financial_expense/income.operating_revenue).label('fina_exp_ratio'), # 质量 财务费用_TTM/营业收入_TTM
            (balance.fixed_assets/balance.equities_parent_company_owners).label('fix_ratio'), # 质量 固定资产/归属母公司股东权益
            ((balance.total_current_assets-balance.total_current_liability)/balance.total_assets).label('networkcap_assets'), # 质量 净运营资本/总资产(其中净营运资本=流动资产-流动负债)
            (balance.cash_equivalents/balance.total_assets).label('moneycap_assets') # 质量 现金/总资产      
        ).filter(valuation.code.in_(feasible_security))
        df1 = get_fundamentals(q, yesterday).set_index('code')
        df2 = get_valuation(feasible_security, end_date=yesterday, count=1, fields='circulating_market_cap')                
        df1 = pd.merge(df1, df2[['code','circulating_market_cap']], on='code', how='left').set_index('code')
        df1['bps'] = df1['total_owner_equities']/df1['circulating_market_cap']
    # Save File
        res_df = df1[['safexp_operrev', 
                      'adm_exp_ratio', 
                      'bps', 
                      'currency_ratio',
                      'asset_turnover',
                      'inv_turnover',
                      'fina_exp_ratio',
                      'fix_ratio',
                      'networkcap_assets',
                      'moneycap_assets']].to_csv('data/9_Quality/{}.csv'.format(str(current_dt)))
    
        

Yesterday=2015-01-09, Today=2015-01-12
Yesterday=2015-01-16, Today=2015-01-19
Yesterday=2015-01-23, Today=2015-01-26
Yesterday=2015-01-30, Today=2015-02-02
Yesterday=2015-02-06, Today=2015-02-09
Yesterday=2015-02-13, Today=2015-02-16
Yesterday=2015-02-17, Today=2015-02-25
Yesterday=2015-02-27, Today=2015-03-02
Yesterday=2015-03-06, Today=2015-03-09
Yesterday=2015-03-13, Today=2015-03-16
Yesterday=2015-03-20, Today=2015-03-23
Yesterday=2015-03-27, Today=2015-03-30
Yesterday=2015-04-03, Today=2015-04-07
Yesterday=2015-04-10, Today=2015-04-13
Yesterday=2015-04-17, Today=2015-04-20
Yesterday=2015-04-24, Today=2015-04-27
Yesterday=2015-04-30, Today=2015-05-04
Yesterday=2015-05-08, Today=2015-05-11
Yesterday=2015-05-15, Today=2015-05-18
Yesterday=2015-05-22, Today=2015-05-25
Yesterday=2015-05-29, Today=2015-06-01
Yesterday=2015-06-05, Today=2015-06-08
Yesterday=2015-06-12, Today=2015-06-15
Yesterday=2015-06-19, Today=2015-06-23
Yesterday=2015-06-26, Today=2015-06-29
Yesterday=2015-07-03, Tod

Yesterday=2019-03-01, Today=2019-03-04
Yesterday=2019-03-08, Today=2019-03-11
Yesterday=2019-03-15, Today=2019-03-18
Yesterday=2019-03-22, Today=2019-03-25
Yesterday=2019-03-29, Today=2019-04-01
Yesterday=2019-04-04, Today=2019-04-08
Yesterday=2019-04-12, Today=2019-04-15
Yesterday=2019-04-19, Today=2019-04-22
Yesterday=2019-04-26, Today=2019-04-29
Yesterday=2019-04-30, Today=2019-05-06
Yesterday=2019-05-10, Today=2019-05-13
Yesterday=2019-05-17, Today=2019-05-20
Yesterday=2019-05-24, Today=2019-05-27
Yesterday=2019-05-31, Today=2019-06-03
Yesterday=2019-06-06, Today=2019-06-10
Yesterday=2019-06-14, Today=2019-06-17
Yesterday=2019-06-21, Today=2019-06-24
Yesterday=2019-06-28, Today=2019-07-01
Yesterday=2019-07-05, Today=2019-07-08
Yesterday=2019-07-12, Today=2019-07-15
Yesterday=2019-07-19, Today=2019-07-22
Yesterday=2019-07-26, Today=2019-07-29
Yesterday=2019-08-02, Today=2019-08-05
Yesterday=2019-08-09, Today=2019-08-12
Yesterday=2019-08-16, Today=2019-08-19
Yesterday=2019-08-23, Tod

Yesterday=2023-04-04, Today=2023-04-06
Yesterday=2023-04-07, Today=2023-04-10
Yesterday=2023-04-14, Today=2023-04-17
Yesterday=2023-04-21, Today=2023-04-24
Yesterday=2023-04-28, Today=2023-05-04
Yesterday=2023-05-05, Today=2023-05-08
Yesterday=2023-05-12, Today=2023-05-15
Yesterday=2023-05-19, Today=2023-05-22
Yesterday=2023-05-26, Today=2023-05-29
Yesterday=2023-06-02, Today=2023-06-05
Yesterday=2023-06-09, Today=2023-06-12
Yesterday=2023-06-16, Today=2023-06-19
Yesterday=2023-06-21, Today=2023-06-26
Yesterday=2023-06-30, Today=2023-07-03
Yesterday=2023-07-07, Today=2023-07-10


# HDF

In [15]:
import os
def getFileSize(filePath, size=0):
    for root, dirs, files in os.walk(filePath):
        for f in files:
            size += os.path.getsize(os.path.join(root, f))
            # print(f)
    return size

size = 0
size += getFileSize('data/clo_5d_60d')
size += getFileSize('data/mom_252d')
size += getFileSize('data/reverse_20d')
size += getFileSize('data/close_max_div_min_20d')
size += getFileSize('data/return_max_20d')
size

204435164

In [16]:
getFileSize('data/Reverse')

142422856